# 02 · Exploratory data analysis

**Pipeline position:** Data cleaning → *EDA / correlation analysis* → Standardization → PCA → Clustering → Interpretation

Covers the five EDA tasks of the assignment:

1. Inspect the dataset (size, missing values, summary statistics, extremes)
2. Distributions, skewness, outliers, and the log transform of GDP per capita
3. Comparison across World Bank income groups
4. Correlations and redundant variables
5. Pairwise scatter plots coloured by income group / region

> EDA uses `countries_filtered.csv` (missing values **not** imputed) so that distributions are not distorted by imputation.
> Income group and region are used here **only to interpret** the data, never as features.

In [ ]:
import sys, json
from pathlib import Path

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "common.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Project root not found. In VS Code use File > Open Folder... and open the Project folder.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.common import (FEATURES, PRETTY, INCOME_ORDER, INCOME_COLORS,
                        apply_transforms, winsorize, signed_log1p)

PROC, FIG = ROOT / "data" / "processed", ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

def savefig(name, **kw):
    plt.tight_layout(**kw)
    plt.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

info = json.loads((PROC / "dataset_info.json").read_text(encoding="utf-8"))
df = pd.read_csv(PROC / "countries_filtered.csv", index_col="code")

# income group restricted to the 4 standard classes (e.g. "Not classified" -> NaN, excluded from group plots)
df["income_plot"] = df["income_group"].where(df["income_group"].isin(INCOME_ORDER))
REGIONS = sorted(df["region"].unique())
REGION_COLORS = dict(zip(REGIONS, sns.color_palette("tab10", len(REGIONS))))

print(f"Year {info['year']} | {len(df)} countries | {len(FEATURES)} indicators | built on {info['run_date']}")

## Task 1 · Inspect the dataset

In [ ]:
print(f"Countries: {df.shape[0]}   Numerical indicators: {len(FEATURES)}   Label columns: region, income_group, lending_type")
print(f"Total missing cells in the 11 indicators: {int(df[FEATURES].isna().sum().sum())} "
      f"of {df[FEATURES].size} ({df[FEATURES].isna().mean().mean() * 100:.1f} %)")

q1, q3 = df[FEATURES].quantile(0.25), df[FEATURES].quantile(0.75)
iqr = q3 - q1
n_out = ((df[FEATURES] < q1 - 1.5 * iqr) | (df[FEATURES] > q3 + 1.5 * iqr)).sum()

summary = df[FEATURES].describe().T
summary["missing"] = df[FEATURES].isna().sum()
summary["missing_%"] = df[FEATURES].isna().mean() * 100
summary["skew"] = df[FEATURES].skew()
summary["max/median"] = summary["max"] / summary["50%"]
summary["IQR-outliers"] = n_out
display(summary.round(2))

In [ ]:
# Extreme observations: three lowest and three highest countries for every indicator
rows = []
for c in FEATURES:
    s = df[c].dropna().sort_values()
    low = ", ".join(f"{df.loc[i, 'name']} ({s[i]:.1f})" for i in s.index[:3])
    high = ", ".join(f"{df.loc[i, 'name']} ({s[i]:.1f})" for i in s.index[::-1][:3])
    rows.append({"indicator": c, "lowest 3": low, "highest 3": high})
display(pd.DataFrame(rows).set_index("indicator"))

**Your observations:** Which indicators have substantial missing data? Which show huge spreads between countries
(look at `max/median` and the IQR-outlier counts)? Which countries keep appearing among the extremes, and are they
data errors or genuine (small open economies, countries in crisis…)?

## Task 2 · Feature distributions, skewness and transformations

In [ ]:
KEY = ["GDP_PC", "Inflation", "Unemployment", "Life_Expectancy", "Internet_Users", "Electricity_Access"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), KEY):
    sns.histplot(df[c].dropna(), bins=30, kde=True, color="steelblue", ax=ax)
    ax.set_title(f"{PRETTY[c]}\nskew = {df[c].skew():.2f}")
    ax.set_xlabel("")
savefig("02_key_distributions")

fig, axes = plt.subplots(1, 6, figsize=(17, 4))
for ax, c in zip(axes, KEY):
    sns.boxplot(y=df[c], color="lightsteelblue", fliersize=3, ax=ax)
    ax.set_title(c); ax.set_ylabel("")
savefig("02_key_boxplots")

In [ ]:
# GDP per capita: raw vs log10  (histogram + normal Q-Q plot)
g = df["GDP_PC"].dropna()
fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
sns.histplot(g, bins=30, kde=True, color="indianred", ax=axes[0])
axes[0].set_title(f"GDP per capita (raw)\nskew = {g.skew():.2f}"); axes[0].set_xlabel("US$")
sns.histplot(np.log10(g), bins=30, kde=True, color="seagreen", ax=axes[1])
axes[1].set_title(f"log10(GDP per capita)\nskew = {np.log10(g).skew():.2f}"); axes[1].set_xlabel("log10(US$)")
stats.probplot(g, dist="norm", plot=axes[2]);          axes[2].set_title("Normal Q-Q: raw")
stats.probplot(np.log10(g), dist="norm", plot=axes[3]); axes[3].set_title("Normal Q-Q: log10")
savefig("02_gdp_log_transform")

### Which transformation for which variable?

Distance-based methods (K-means, Ward linkage, PCA) are very sensitive to heavy tails: a handful of extreme values can
dominate a standardised feature. For every skewed or outlier-prone candidate we compare the skewness after
(a) clipping at the 1st/99th percentile (winsorising), (b) a signed log transform, `sign(x)·ln(1+|x|)`, which handles
negative values such as deflation, and (c) a plain `log10` where all values are positive.

In [ ]:
def skew_table(cols):
    out = {}
    for c in cols:
        s = df[c].dropna()
        out[c] = {"raw": s.skew(),
                  "winsor 1-99%": winsorize(s).skew(),
                  "signed log1p": signed_log1p(s).skew(),
                  "log10": np.log10(s).skew() if (s > 0).all() else np.nan}
    return pd.DataFrame(out).T

display(skew_table(["GDP_PC", "Inflation", "Trade", "GDP_Growth", "Unemployment", "Capital_Formation"]).round(2))

**Transformations applied downstream (defined once in `src/common.py`, function `apply_transforms`):**

| Variable | Transformation | Reason |
|---|---|---|
| GDP per capita | `log10(x)` | strongly right-skewed over several orders of magnitude; log makes the distribution close to symmetric and turns multiplicative gaps into additive ones |
| Inflation | `sign(x)·ln(1+|x|)` | extreme hyper-inflation cases; can be negative, so a plain log is impossible |
| Trade, GDP growth | winsorise at 1st / 99th percentile | a few extreme observations (very small open economies, post-crisis rebounds) |
| All others | none | roughly symmetric or bounded in [0, 100] |

*Confirm these against the table above — if another option reduces the skewness better for some variable,
change the lists in `src/common.py` and note why in your report.*

In [ ]:
X_t = apply_transforms(df)
cmp_skew = pd.DataFrame({"before": df[FEATURES].skew().values, "after": X_t.skew().values}, index=X_t.columns)
display(cmp_skew.round(2).T)

ax = cmp_skew.plot.barh(figsize=(8, 5.5), color=["#bbbbbb", "#2c7fb8"])
ax.axvline(0, color="k", lw=0.8); ax.axvline(1, color="r", ls=":", lw=0.8); ax.axvline(-1, color="r", ls=":", lw=0.8)
ax.set_title("Skewness before vs after transformation (dotted: |skew| = 1)"); ax.invert_yaxis()
savefig("02_skew_before_after")

## Task 3 · Comparison across World Bank income groups

The income group is used **only as an interpretive label**. (Economies without a standard income class are left out of these plots.)

In [ ]:
CMP = ["GDP_PC", "Internet_Users", "Life_Expectancy", "Electricity_Access"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, c in zip(axes.ravel(), CMP):
    sns.boxplot(data=df, x="income_plot", y=c, order=INCOME_ORDER, hue="income_plot", hue_order=INCOME_ORDER,
                palette=INCOME_COLORS, legend=False, fliersize=0, ax=ax)
    sns.stripplot(data=df, x="income_plot", y=c, order=INCOME_ORDER, color="k", size=2.5, alpha=0.35, ax=ax)
    if c == "GDP_PC":
        ax.set_yscale("log")
    ax.set_title(PRETTY[c]); ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(axis="x", labelrotation=12)
savefig("02_income_group_comparison")

print("Countries per income group:")
display(df["income_plot"].value_counts().reindex(INCOME_ORDER).to_frame("n"))
print("Median by income group:")
display(df.groupby("income_plot")[CMP].median().reindex(INCOME_ORDER).round(1))

**Your observations:** Which indicators separate the income groups cleanly and which overlap? Is there a ceiling effect
(e.g. electricity access ≈ 100 % in most groups)? If an indicator barely separates the groups, what does that tell you about
how much that variable can contribute to a clustering that "recovers" income groups?

## Task 4 · Correlations and redundant variables

Pearson correlations are computed on the **transformed** features (the ones used for clustering); Spearman (rank) correlations
on the **raw** values serve as a check that is robust to outliers and monotone transformations.

In [ ]:
pear = X_t.corr(method="pearson")
spear = df[FEATURES].corr(method="spearman")
tri = np.triu(np.ones_like(pear, dtype=bool), k=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(pear, mask=tri, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, cbar_kws={"shrink": 0.7}, ax=axes[0])
axes[0].set_title("Pearson - transformed features")
sns.heatmap(spear, mask=tri, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, cbar_kws={"shrink": 0.7}, ax=axes[1])
axes[1].set_title("Spearman - raw features")
savefig("02_correlation_heatmaps")

def high_pairs(C, thr=0.8):
    pairs = [(a, b, C.loc[a, b]) for i, a in enumerate(C.columns) for b in C.columns[i + 1:] if abs(C.loc[a, b]) >= thr]
    return pd.DataFrame(pairs, columns=["variable 1", "variable 2", "r"]).sort_values("r", key=abs, ascending=False)

print("Pairs with |Pearson r| >= 0.8 (candidates for redundancy):")
display(high_pairs(pear).round(2))

# Variance inflation factors: how well is each variable explained by all the others? (complete cases only)
cc = X_t.dropna()
R = np.corrcoef(cc.values, rowvar=False)
vif = pd.Series(np.diag(np.linalg.inv(R)), index=cc.columns, name="VIF").sort_values(ascending=False)
print(f"VIF (computed on {len(cc)} complete-case countries; > 5 suggests strong redundancy, > 10 severe):")
display(vif.round(1).to_frame().T)

**Your observations:** Which blocks of variables move together (income ↔ health ↔ internet ↔ electricity ↔ education ↔ urbanisation)?
Which macro variables (growth, inflation, unemployment, trade, investment) are nearly uncorrelated with everything else?
Highly correlated variables are not removed here — **PCA handles redundancy by design** — but note that a strong
correlated block will dominate PC1 and therefore the distances used in clustering.

## Task 5 · Pairwise relationships

GDP per capita is on a log axis in all panels (consistent with the transformation chosen above).
Top figure: coloured by income group. Bottom figure: coloured by geographical region.

In [ ]:
# (x, y, log-x?)   GDP per capita on the x-axis for panels 1, 2, 4; education on x for panel 3
PANELS = [("GDP_PC", "Life_Expectancy", True),
          ("GDP_PC", "Internet_Users", True),
          ("Secondary_Enrollment", "Life_Expectancy", False),
          ("GDP_PC", "Urban_Pop", True)]

def scatter_grid(hue, hue_order, palette, fname, ncol):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (x, y, logx) in zip(axes.ravel(), PANELS):
        sns.scatterplot(data=df, x=x, y=y, hue=hue, hue_order=hue_order, palette=palette,
                        s=45, alpha=0.8, edgecolor="white", linewidth=0.4, ax=ax)
        if logx:
            ax.set_xscale("log")
        ax.set_xlabel(PRETTY[x]); ax.set_ylabel(PRETTY[y])
        ax.set_title(f"{PRETTY[y].split(' (')[0]} vs {PRETTY[x].split(' (')[0]}")
        ax.get_legend().remove()
    handles = [plt.Line2D([], [], marker="o", ls="", color=palette[h], label=h) for h in hue_order]
    fig.legend(handles=handles, loc="lower center", ncol=ncol, frameon=False)
    savefig(fname, rect=[0, 0.05, 1, 1])

scatter_grid("income_plot", INCOME_ORDER, INCOME_COLORS, "02_scatter_income", ncol=4)
scatter_grid("region", REGIONS, REGION_COLORS, "02_scatter_region", ncol=4)

**Your observations:** Is the relationship between log GDP per capita and life expectancy linear or does it saturate at high income?
Are there visible groups or gaps (which would suggest natural clusters), or a continuum? Which regions sit systematically
above or below the trend (e.g. Sub-Saharan Africa vs Latin America)? Which individual countries lie far from the general pattern?

## Summary to carry forward

Fill this in from your observations — it feeds directly into the report:

* **Dataset:** year, number of countries, indicators with most missing data, and the strategy used
* **Transformations:** log10 GDP per capita, signed-log inflation, winsorised trade and growth — with the skewness numbers
* **Redundancy:** which variable blocks are strongly correlated
* **Structure seen:** continuum or visible groups; which indicators separate income groups best

```bash
git add notebooks/02_eda.ipynb reports/figures
git commit -m "EDA: distributions, income-group comparison, correlations, scatter plots"
git push
```

**Next:** `03_pca.ipynb` — standardise, PCA, explained variance, loadings.